# Phase-0 pilot — Conformal Burden v2 (ERM + DFR on Waterbirds), then **STOP**

Pre-registered campaign v2, spec **§1**. Runs the **minimal pilot only**: **ERM** and **DFR**
(last-layer retrain) on **Waterbirds**, on frozen CLIP ViT-B/32 features (encoded once, cached),
over **3 seeds**. Reports each model's **worst-group accuracy** + a **sample cross-group
conformity-score divergence (APS)**, then **HALTS for human review.**

**This notebook does NOT run the H1/H2/H3 grid.** Phase-0-then-STOP is a hard discipline.

**The STOP gate (HARD).** After the numbers print, the gate cell checks them against the published
reference: **DFR worst-group ≈ 0.86–0.92**, **ERM worst-group ≈ 0.60–0.75**. If **DFR ≈ ERM** or
**near chance**, the pipeline is broken → it writes `BLOCKERS.md` and halts. Whatever the numbers
are, report them honestly — do not re-tune to recover a desired result.

Run top-to-bottom on a **GPU** runtime (`Runtime → Change runtime type → GPU`).

## 0. Parameters — **EDIT THESE**

In [ ]:
# ===================== EDIT THESE =====================
REPO_SOURCE    = "git"          # "git" or "drive"
REPO_URL       = "https://github.com/octadion/vgscp.git"   # EDIT (private: https://<TOKEN>@github.com/<user>/vgscp.git)
REPO_BRANCH    = "main"
REPO_DRIVE_ZIP = "/content/drive/MyDrive/vgscp.zip"           # used only if REPO_SOURCE=="drive"

DRIVE_CACHE    = "/content/drive/MyDrive/vgscp_cache"  # datasets + CLIP feature cache persisted here
SEEDS          = 3                                     # spec section 1: 3 training seeds for the pilot

# Dataset URLs - EDIT IF URL CHANGES
WATERBIRDS_URL = "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
CUB_URL        = "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz"
# ======================================================
import os, time, subprocess, sys
def sh(cmd, **kw):
    print("$", cmd); return subprocess.run(cmd, shell=True, **kw)

## 1. GPU check + install

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("[WARN] No GPU - CLIP encode will be slow. Set Runtime->GPU.")
subprocess.run("pip -q install open_clip_torch ftfy regex tqdm pyyaml scikit-learn scipy pandas matplotlib", shell=True)

## 2. Mount Drive (persist datasets + CLIP feature cache across restarts)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_CACHE, exist_ok=True)
print("cache dir:", DRIVE_CACHE)

## 3. Get the repo

In [ ]:
REPO_DIR = "/content/vgscp"
if REPO_SOURCE == "git":
    sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
else:
    sh(f"rm -rf {REPO_DIR} && mkdir -p {REPO_DIR} && unzip -q {REPO_DRIVE_ZIP} -d {REPO_DIR}")
    subs = [d for d in os.listdir(REPO_DIR) if os.path.isdir(f"{REPO_DIR}/{d}")]
    if len(subs) == 1 and not os.path.exists(f"{REPO_DIR}/scripts"):
        inner = f"{REPO_DIR}/{subs[0]}"; sh(f"shopt -s dotglob && mv {inner}/* {REPO_DIR}/ && rmdir {inner}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print("repo:", os.getcwd())

## 4. Datasets — download (cached to Drive) + extract, set env vars

In [ ]:
def fetch(url, drive_name, extract_to):
    os.makedirs(extract_to, exist_ok=True)
    tarball = os.path.join(DRIVE_CACHE, drive_name)
    if not os.path.exists(tarball):
        sh(f"wget -q -O '{tarball}' '{url}'")
    else:
        print("cached tarball:", tarball)
    sh(f"tar -xzf '{tarball}' -C '{extract_to}'")
    return extract_to

fetch(WATERBIRDS_URL, "waterbirds.tar.gz", "/content/data/waterbirds")
fetch(CUB_URL, "CUB_200_2011.tgz", "/content/data/cub")
os.environ["WATERBIRDS_ROOT"] = "/content/data/waterbirds"
os.environ["CUB_ROOT"] = "/content/data/cub"
# point the repo CLIP feature cache at Drive so re-runs skip re-encoding
sh("rm -rf results/cache_clip"); os.makedirs("results", exist_ok=True)
os.makedirs(f"{DRIVE_CACHE}/clip", exist_ok=True)
sh(f"ln -s {DRIVE_CACHE}/clip results/cache_clip")
print("WATERBIRDS_ROOT=", os.environ["WATERBIRDS_ROOT"]); print("CUB_ROOT=", os.environ["CUB_ROOT"])

## 5. Encode + cache CLIP features ONCE
Reuses `experiments.real_data.load_real_bundle` (audit-verified live). The features are the
**composited** Waterbirds images; the bundle exposes binary `y` (land/water bird), `place`
(background), `group_id` (the 4 Waterbirds groups) per split. `assert_l2_normalized` is the
section-2a guard reused from the live pipeline.

In [ ]:
t = time.time()
from config_util import load_config
from experiments.real_data import load_real_bundle, assert_l2_normalized
cfg = load_config("configs/cub200_frontier.yaml")
bundle = load_real_bundle(cfg, seed=0)
print("feature shapes:", {k: v.shape for k, v in bundle.features.items()})
# section 2a: every composited split feeding a head must be L2-normalized (in-domain).
for sp in ("train", "d_learn", "d_test"):
    assert_l2_normalized(bundle.features[sp], tag=f"composited {sp} features")
print("L2-normalized assert PASSED for train / d_learn / d_test")
print(f"[encode+cache] wall={(time.time()-t)/60:.1f} min (cached to Drive)")

## 6. Build Phase-0 data (binary Waterbirds task, in-domain)
ERM fits on the composited `train` split. DFR (last-layer retrain) fits on the **held-out**
composited `d_learn` split, group-balanced — the textbook DFR reweighting split. Both are the
**same composited distribution** as the `d_test` we evaluate on (section 2 in-domain train+test).

In [ ]:
from study_robust_train.phase0 import Phase0Data
import numpy as np

data = Phase0Data(
    feats_train=bundle.features["train"], y_train=bundle.y["train"], place_train=bundle.place["train"],
    feats_test=bundle.features["d_test"], y_test=bundle.y["d_test"], place_test=bundle.place["d_test"],
    n_classes=2, synthetic=False,
)
# Held-out reweighting split for DFR (in-domain composited).
dfr_fit = (bundle.features["d_learn"], bundle.y["d_learn"],
           (2 * bundle.y["d_learn"].astype(int) + bundle.place["d_learn"].astype(int)))

def grp_counts(y, place):
    g = 2 * np.asarray(y).astype(int) + np.asarray(place).astype(int)
    return {int(k): int((g == k).sum()) for k in range(4)}
print("train   group counts (g=2y+place):", grp_counts(bundle.y["train"], bundle.place["train"]))
print("d_learn group counts:", grp_counts(bundle.y["d_learn"], bundle.place["d_learn"]))
print("d_test  group counts:", grp_counts(bundle.y["d_test"], bundle.place["d_test"]))

## 7. Run Phase-0 — ERM + DFR, 3 seeds -> worst-group accuracy + sample APS divergence

In [ ]:
import json
from study_robust_train.phase0 import run_phase0, format_report

out = run_phase0(data, seeds=tuple(range(SEEDS)), dfr_subsets=10, dfr_fit=dfr_fit)
print(format_report(out))

# Save raw results (json + tidy CSV) for the repro bundle.
os.makedirs("results/phase0", exist_ok=True)
serializable = {
    "synthetic": out["synthetic"], "seeds": out["seeds"], "n_classes": out["n_classes"],
    "aggregate": out["aggregate"], "dfr_wg_reference": out["dfr_wg_reference"],
    "erm_wg_reference": out["erm_wg_reference"],
    "per_run": [{"method": r.method, "seed": r.seed, "worst_group": r.worst_group,
                 "worst_group_acc": r.worst_group_acc, "overall_acc": r.overall_acc,
                 "per_group_acc": r.per_group_acc, "aps_wasserstein1": r.aps_wasserstein1,
                 "aps_ks_stat": r.aps_ks_stat, "aps_ks_pvalue": r.aps_ks_pvalue} for r in out["per_run"]],
}
with open("results/phase0/phase0_results.json", "w", encoding="utf-8") as f:
    json.dump(serializable, f, indent=2)

import pandas as pd
df = pd.DataFrame(serializable["per_run"]).drop(columns=["per_group_acc"])
df.to_csv("results/phase0/phase0_per_run.csv", index=False)
print("\nsaved results/phase0/phase0_results.json + phase0_per_run.csv")
df

## 8. STOP GATE — check vs published reference, then HALT (spec section 1)
**DFR worst-group ≈ 0.86–0.92, ERM worst-group ≈ 0.60–0.75.** If **DFR ≈ ERM** or **near chance**,
the pipeline is broken → write `BLOCKERS.md` and stop; **fix before any grid.** Either way, this is
where the notebook ends — the next step is the researcher's review, not the grid.

In [ ]:
erm_wg = out["aggregate"]["ERM"]["worst_group_acc_mean"]
dfr_wg = out["aggregate"]["DFR"]["worst_group_acc_mean"]
DFR_LO, DFR_HI = out["dfr_wg_reference"]
ERM_LO, ERM_HI = out["erm_wg_reference"]

near_chance     = dfr_wg < 0.55                 # binary task chance ~0.5
dfr_collapsed   = (dfr_wg - erm_wg) < 0.03      # DFR ~ ERM => no robustness gained
dfr_below_ref   = dfr_wg < DFR_LO               # below published range (softer warning)
PIPELINE_BROKEN = bool(near_chance or dfr_collapsed)

print("=" * 78)
print(f"ERM worst-group acc = {erm_wg:.3f}  (ref {ERM_LO}-{ERM_HI})")
print(f"DFR worst-group acc = {dfr_wg:.3f}  (ref {DFR_LO}-{DFR_HI})")
print(f"DFR - ERM           = {dfr_wg - erm_wg:+.3f}")
print("=" * 78)

if PIPELINE_BROKEN:
    why = []
    if near_chance:   why.append(f"DFR worst-group {dfr_wg:.3f} is near chance (<0.55)")
    if dfr_collapsed: why.append(f"DFR ({dfr_wg:.3f}) did not improve over ERM ({erm_wg:.3f}); gap <0.03")
    blockers = (
        "# BLOCKERS.md - Phase-0 gate FAILED (spec section 1)\n\n"
        f"DFR worst-group accuracy did NOT match the published Waterbirds range ({DFR_LO}-{DFR_HI}).\n\n"
        f"- ERM worst-group acc: {erm_wg:.3f} (ref {ERM_LO}-{ERM_HI})\n"
        f"- DFR worst-group acc: {dfr_wg:.3f} (ref {DFR_LO}-{DFR_HI})\n"
        f"- DFR - ERM: {dfr_wg - erm_wg:+.3f}\n\n"
        "## Why this is a blocker\n" + "\n".join(f"- {w}" for w in why) + "\n\n"
        "## Do NOT proceed to the grid. Fix the pipeline first. Candidate causes:\n"
        "- Feature extraction: non-canonical CLIP transform / un-normalized features.\n"
        "- Group labels (place) misaligned with the composited images.\n"
        "- DFR reweighting split too small or missing a group (check d_learn group counts above).\n"
        "- Label/feature cache desync.\n\n"
        "Per the spec: do not re-tune to recover a desired result; fix the mechanism and re-run Phase-0.\n"
    )
    with open("BLOCKERS.md", "w", encoding="utf-8") as f:
        f.write(blockers)
    print(blockers)
    raise SystemExit("PHASE-0 GATE FAILED - BLOCKERS.md written. Notebook halted; do not run the grid.")
else:
    verdict = "PASS" if not dfr_below_ref else "PASS-with-CAVEAT (DFR below published range, but clearly > ERM)"
    print(f"GATE {verdict}: DFR ({dfr_wg:.3f}) materially exceeds ERM ({erm_wg:.3f}).")
    if dfr_below_ref:
        print(f"  CAVEAT: DFR {dfr_wg:.3f} < published low bound {DFR_LO}. Flag for human judgment "
              f"(recipe detail, e.g. C / d_learn size), not necessarily broken.")
    print("\nPHASE-0 COMPLETE - STOP for human review. Do NOT proceed to the H1/H2/H3 grid.")

## 9. STOP — Phase-0 ends here
Hand the worst-group accuracies + APS divergence above to the researcher. **The H1/H2/H3 grid is
intentionally not in this notebook.** It runs only after the researcher confirms the pilot looks
sane (DFR in the published range, ERM lower) per spec section 1.